In [ ]:
#https://ryan-reilly.medium.com/gridsearch-pipelines-of-multiple-models-on-multiclass-classification-e9124b6ea2e3
#https://scikit-learn.org/stable/tutorial/statistical_inference/putting_together.html

#Feature selection
#https://scikit-learn.org/stable/modules/feature_selection.html

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
abandono = pd.read_csv('./Abandono/abandono.csv',index_col=0)
abandono

In [ ]:
grades_per_exam = pd.read_csv('./Grades/grades_per_exam.csv',index_col=0)
grades_per_exam

# Pipeline + GridSearchCV


## **Abandono**

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn import svm

In [ ]:
import sklearn
sklearn.metrics.get_scorer_names()

In [ ]:
def abandono_pipeline(w, indicadores):
  w_ind = indicadores.copy()
  w_ind['passed'] = abandono['curso_superado']
  w_ind['dropped_out'] = abandono['abandono']

  w_ind['CFA'] = w_ind['CFA'].replace('0 problems attempted',0)
  w_ind['avg_grade'] = w_ind['avg_grade'].replace('0 problems attempted',0)
  w_ind['avg_attempts'] = w_ind['avg_attempts'].replace('0 problems attempted',0)
  w_ind['dropped_out'] = w_ind['dropped_out'].replace('No',0).replace('Sí',1)
  w_ind['passed'] = w_ind['passed'].replace('No',0).replace('Sí',1)

  X = w_ind.drop(['dropped_out','passed'],axis=1)
  y = w_ind['dropped_out']

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

  pipe_lr = Pipeline([('scl', StandardScaler()),
                    ('LR', LogisticRegression(random_state=42))])
  pipe_dt = Pipeline([('scl', StandardScaler()),
                    ('DT',DecisionTreeClassifier(random_state=42))])
  pipe_rf = Pipeline([('scl', StandardScaler()),
                      ('RF',RandomForestClassifier(random_state=42))])
  pipe_knn = Pipeline([('scl', StandardScaler()),
                      ('KNN', KNeighborsClassifier())])
  pipe_svm = Pipeline([('scl', StandardScaler()),
                      ('SVM', svm.SVC(random_state=42))])
  pipe_xgb = Pipeline([('scl', StandardScaler()),
                      ('XGB', XGBClassifier(random_state=42))])

  param_range = [1, 2, 3, 5]
  param_range_fl = [1.0, 0.5, 0.1]
  n_estimators = [50,75,100]
  learning_rates = [.1,.2,.3]
  lr_param_grid = [{'LR__penalty': ['l1', 'l2'],
                   'LR__C': param_range_fl,
                   'LR__solver': ['liblinear']}]
  dt_param_grid = [{'DT__criterion': ['gini', 'entropy'],
                    'DT__min_samples_leaf': param_range,
                    'DT__max_depth': param_range,
                    'DT__min_samples_split': param_range[1:]}]
  rf_param_grid = [{'RF__min_samples_leaf': param_range,
                    'RF__max_depth': param_range,
                    'RF__min_samples_split': param_range[1:]}]
  knn_param_grid = [{'KNN__n_neighbors': param_range,
                    'KNN__weights': ['uniform', 'distance'],
                    'KNN__metric': ['euclidean', 'manhattan']}]
  svm_param_grid = [{'SVM__kernel': ['linear', 'rbf'],
                      'SVM__C': param_range}]
  xgb_param_grid = [{'XGB__learning_rate': learning_rates,
                      'XGB__max_depth': param_range,
                      'XGB__min_child_weight': param_range[:2],
                      'XGB__subsample': param_range_fl,
                      'XGB__n_estimators': n_estimators}]

  lr_grid_search = GridSearchCV(estimator=pipe_lr,
        param_grid=lr_param_grid,
        scoring='roc_auc',
        cv=3)
  dt_grid_search = GridSearchCV(estimator=pipe_dt,
        param_grid=dt_param_grid,
        scoring='roc_auc',
        cv=3)
  rf_grid_search = GridSearchCV(estimator=pipe_rf,
          param_grid=rf_param_grid,
          scoring='roc_auc',
          cv=3)
  knn_grid_search = GridSearchCV(estimator=pipe_knn,
          param_grid=knn_param_grid,
          scoring='roc_auc',
          cv=3)
  svm_grid_search = GridSearchCV(estimator=pipe_svm,
          param_grid=svm_param_grid,
          scoring='roc_auc',
          cv=3)
  xgb_grid_search = GridSearchCV(estimator=pipe_xgb,
          param_grid=xgb_param_grid,
          scoring='roc_auc',
          cv=3)
  grids = [lr_grid_search, dt_grid_search, rf_grid_search, knn_grid_search, svm_grid_search, xgb_grid_search]
  names = ['lr','dt', 'rf', 'knn', 'svm', 'xgb']
  n = 0
  for pipe in grids:
    print(names[n])
    pipe.fit(X_train,y_train)
    n+=1

  aucs = {}
  grid_dict = {0: 'Logistic Regression', 1: 'Decision Trees',
             2: 'Random Forest', 3: 'K-Nearest Neighbors',
             4: 'Support Vector Machines', 5: 'XGBoost'}
  for i, model in enumerate(grids):
    aucs[grid_dict[i]] = model.score(X_test,y_test)

  return aucs

In [ ]:
aucs = {}
for w in range(1,8):
  ind =  pd.read_csv('./Modelos/Cummulative/Indicadores/indicadores_week1_{}.csv'.format(w),index_col=0)
  conf_ind = ind.loc[grades_per_exam.index]
  print('\nWeek', w)
  auc =  abandono_pipeline(w, indicadores = conf_ind)
  aucs['Week {}'.format(str(w))] = auc
  print(aucs['Week {}'.format(str(w))])

In [ ]:
fig = plt.subplots(2,3, figsize=(12,5),constrained_layout = True)

models = ['Logistic Regression','Decision Trees', 'Random Forest','K-Nearest Neighbors','Support Vector Machines', 'XGBoost']

auc_model = {}
for m in models:
  auc_model[m] = {}
  for i in range(1,8):
    auc_model[m][i] = aucs['Week {}'.format(i)][m]
print(auc_model)

i=0
for m in models:
  x, y = zip(*auc_model[m].items())
  ax = plt.subplot(2, 3, i + 1)
  plt.plot(x, y, marker='o')
  plt.ylim(0.5,1)
  plt.xlabel('Week')
  plt.title(f'AUC {m}')
  i+=1

In [ ]:
aucs

# **Nota final**

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

In [ ]:
def nota_final_pipeline(w, indicadores):
  w_ind = indicadores.copy()
  w_ind['grade'] = grades_per_exam['Course grade']
  w_ind['grade'] = w_ind['grade'].fillna(0)

  w_ind['CFA'] = w_ind['CFA'].replace('0 problems attempted',0)
  w_ind['avg_grade'] = w_ind['avg_grade'].replace('0 problems attempted',0)
  w_ind['avg_attempts'] = w_ind['avg_attempts'].replace('0 problems attempted',0)

  X = w_ind.drop(['grade'],axis=1)
  y = w_ind['grade']

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

  pipe_r = Pipeline([('scl', StandardScaler()),
                    ('R', Ridge(random_state=42))])
  pipe_dt = Pipeline([('scl', StandardScaler()),
                    ('DT',DecisionTreeRegressor())])
  pipe_rf = Pipeline([('scl', StandardScaler()),
                      ('RF',RandomForestRegressor())])
  pipe_knn = Pipeline([('scl', StandardScaler()),
                      ('KNN', KNeighborsRegressor())])
  pipe_svm = Pipeline([('scl', StandardScaler()),
                      ('SVM', svm.SVR())])
  pipe_xgb = Pipeline([('scl', StandardScaler()),
                      ('XGB', XGBRegressor())])

  param_range = [1, 2, 3, 5]
  param_range_fl = [1.0, 0.5, 0.1]
  n_estimators = [50,75,100]
  learning_rates = [.1,.2,.3]
  alpha = [0,0.1,1,5,10]

  r_param_grid = [{'R__alpha':alpha,
                    'R__fit_intercept':[True, False]}]
  dt_param_grid = [{'DT__min_samples_leaf': param_range,
                    'DT__max_depth': param_range,
                    'DT__min_samples_split': param_range[1:]}]
  rf_param_grid = [{'RF__min_samples_leaf': param_range,
                    'RF__max_depth': param_range,
                    'RF__min_samples_split': param_range[1:]}]
  knn_param_grid = [{'KNN__n_neighbors': param_range,
                    'KNN__weights': ['uniform', 'distance'],
                    'KNN__metric': ['euclidean', 'manhattan']}]
  svm_param_grid = [{'SVM__kernel': ['linear', 'rbf'],
                      'SVM__C': param_range}]
  xgb_param_grid = [{'XGB__learning_rate': learning_rates,
                      'XGB__max_depth': param_range,
                      'XGB__min_child_weight': param_range[:2],
                      'XGB__subsample': param_range_fl,
                      'XGB__n_estimators': n_estimators}]

  r_grid_search = GridSearchCV(estimator=pipe_r,
        param_grid=r_param_grid,
        scoring='neg_mean_squared_error',
        cv=3)
  dt_grid_search = GridSearchCV(estimator=pipe_dt,
        param_grid=dt_param_grid,
        scoring='neg_mean_squared_error',
        cv=3)
  rf_grid_search = GridSearchCV(estimator=pipe_rf,
          param_grid=rf_param_grid,
          scoring='neg_mean_squared_error',
          cv=3)
  knn_grid_search = GridSearchCV(estimator=pipe_knn,
          param_grid=knn_param_grid,
          scoring='neg_mean_squared_error',
          cv=3)
  svm_grid_search = GridSearchCV(estimator=pipe_svm,
          param_grid=svm_param_grid,
          scoring='neg_mean_squared_error',
          cv=3)
  xgb_grid_search = GridSearchCV(estimator=pipe_xgb,
          param_grid=xgb_param_grid,
          scoring='neg_mean_squared_error',
          cv=3)
  grids = [r_grid_search, dt_grid_search, rf_grid_search, knn_grid_search, svm_grid_search, xgb_grid_search]
  names = ['r','dt', 'rf', 'knn', 'svm', 'xgb']
  n = 0
  for pipe in grids:
    print(names[n])
    pipe.fit(X_train,y_train)
    n+=1

  mses = {}
  grid_dict = {0: 'Ridge Regression', 1:'Decision Trees',
             2: 'Random Forest', 3: 'K-Nearest Neighbors',
             4: 'Support Vector Machines', 5: 'XGBoost'}
  for i, model in enumerate(grids):
    mses[grid_dict[i]] = -(model.score(X_test,y_test))

  return mses

In [ ]:
mses = {}
for w in range(1,8):
  ind =  pd.read_csv('./Modelos/Cummulative/Indicadores/indicadores_week1_{}.csv'.format(w),index_col=0)
  conf_ind = ind.loc[grades_per_exam.index]
  print('\nWeek', w)
  mse =  nota_final_pipeline(w, indicadores = conf_ind)
  mses['Week {}'.format(str(w))] = mse
  print(mses['Week {}'.format(str(w))])

In [ ]:
fig = plt.subplots(2,3, figsize=(12,5),constrained_layout = True)

models = ['Ridge Regression','Decision Trees', 'Random Forest','K-Nearest Neighbors','Support Vector Machines','XGBoost']

mse_model = {}
for m in models:
  mse_model[m] = {}
  for i in range(1,8):
    mse_model[m][i] = mses['Week {}'.format(i)][m]
print(mse_model)

i=0
for m in models:
  x, y = zip(*mse_model[m].items())
  ax = plt.subplot(2, 3, i + 1)
  plt.plot(x, y, marker='o')
  plt.ylim(0,0.09)
  plt.xlabel('Week')
  plt.title(f'AUC {m}')
  i+=1

In [ ]:
mses